# Moduł 10: Projekt końcowy — pełny potok inżynierii danych

To ostatni moduł kursu -- i zamiast uczyć się nowego narzędzia, **połączymy wszystkie
poprzednie w jeden, działający potok danych end-to-end**. Zbudujemy dokładnie taką
architekturę, jaką naszkicowaliśmy koncepcyjnie w Module 1:

```
  dane wsadowe                    strumień zdarzen (Modul 8-9)
  (ERP/CRM)                       nowe zamowienia w "czasie rzeczywistym"
      |                                    |
      v                                    v
  +-------------------------------------------------+
  |               strefa "raw" (DuckDB)              |
  +-------------------------------------------------+
      |
      v   dbt: staging -> snapshot (SCD2) -> model gwiazdy   (Modul 4-7)
  +-------------------------------------------------+
  |     dim_klienci (SCD2) | dim_produkty | dim_data |
  |                fact_sprzedaz                     |
  +-------------------------------------------------+
      |
      v
  raport analityczny

  Calosc orkiestrowana przez Apache Airflow (Modul 2-3)
```

Zbudujemy:
1. **Warstwę surowych danych i model gwiazdy** w dbt -- z wymiarem klientów śledzonym jako
   SCD2 (Moduł 6-7).
2. **Komponent streamingowy** (Spark Structured Streaming, Moduł 9), który dokłada nowe
   zamówienia do surowej warstwy w niemal czasie rzeczywistym.
3. **DAG w Airflow** (Moduł 2-3), który orkiestruje cały powyższy potok -- od wygenerowania
   danych, przez przetworzenie strumienia, po transformacje dbt i finalny raport.

Na koniec uruchomimy cały DAG jednym poleceniem i zweryfikujemy wynik, odpytując
zbudowaną hurtownię.

> 💡 **Jak czytać ten moduł**
>
> To dłuższy, bardziej złożony notebook niż poprzednie -- ale każdy fragment kodu wykorzystuje wyłącznie rzeczy, które już znasz z Modułów 1-9. Nowością jest tylko to, *jak układamy je razem* w jeden spójny system.

## 1. Przygotowanie środowiska

Tak jak w Modułach 2-3, uruchomimy Airflow w izolowanym `AIRFLOW_HOME`. Będziemy też
potrzebować dbt (Moduł 4-7) oraz Sparka (Moduł 9) -- każde z tych narzędzi zadziała w
swoim własnym, odpowiednim środowisku uruchomieniowym.

In [ ]:
import os

AIRFLOW_HOME = os.path.abspath("airflow_home")
AIRFLOW_BIN = "/tmp/airflow_venv/bin/airflow"
os.makedirs(f"{AIRFLOW_HOME}/dags", exist_ok=True)

env = os.environ.copy()
env["AIRFLOW_HOME"] = AIRFLOW_HOME
env["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
env["AIRFLOW__LOGGING__LOGGING_LEVEL"] = "WARNING"

import subprocess


def airflow(*args, check=True):
    wynik = subprocess.run([AIRFLOW_BIN, *args], env=env, capture_output=True, text=True)
    print(wynik.stdout)
    if wynik.returncode != 0:
        print(wynik.stderr[-4000:])
        if check:
            raise RuntimeError(f"airflow {' '.join(args)} zakonczyl sie kodem {wynik.returncode}")
    return wynik


airflow("db", "migrate")

> ⚡ **Trzy różne interpretery Pythona -- świadomie**
>
> Ten projekt używa trzech oddzielnych środowisk Pythona jednocześnie: **głównego** (to, w którym działa ten notebook -- ma zainstalowane `dbt`, `duckdb`, `pandas`), **izolowanego Airflow** (`/tmp/airflow_venv` -- ma tylko `apache-airflow`) i **izolowanego Sparka** (`/tmp/spark_venv` -- ma `pyspark` + `duckdb`). To dokładnie odzwierciedla realne wdrożenia: scheduler Airflow nie musi mieć zainstalowanych wszystkich zależności każdego zadania -- jego rolą jest tylko **wywołanie** właściwego narzędzia (przez `BashOperator`) we właściwym środowisku, nie wykonanie pracy samemu.

## 2. Projekt dbt: model gwiazdy z wymiarem SCD2

Budujemy projekt dbt niemal identyczny z tym z Modułów 4-7, ale tym razem z jedną ważną
zmianą: tabela `raw.zamowienia` (i `raw.pozycje_zamowien`) będzie miała swój **strumieniowy
odpowiednik** (`raw.zamowienia_streaming`, `raw.pozycje_zamowien_streaming`) -- pusty na
starcie, wypełniany później przez zadanie Structured Streaming. Modele `staging` łączą oba
źródła (`UNION ALL`) -- z punktu widzenia reszty potoku to jedno, spójne źródło zamówień,
niezależnie od tego, czy dane przyszły wsadowo, czy w czasie rzeczywistym.

In [ ]:
PROJEKT_DIR = os.path.abspath(".")
DBT_DIR = f"{PROJEKT_DIR}/dbt_final"
for poddir in ("models/staging", "models/gwiazda", "snapshots", "seeds", "macros", "tests"):
    os.makedirs(f"{DBT_DIR}/{poddir}", exist_ok=True)

DUCKDB_PATH = f"{DBT_DIR}/kurs_final.duckdb"

with open(f"{DBT_DIR}/dbt_project.yml", "w") as f:
    f.write(
        "name: kurs_final\n"
        "version: '1.0'\n"
        "profile: kurs_final\n"
        "model-paths: [\"models\"]\n"
        "seed-paths: [\"seeds\"]\n"
        "snapshot-paths: [\"snapshots\"]\n"
    )

with open(f"{DBT_DIR}/profiles.yml", "w") as f:
    f.write(
        "kurs_final:\n"
        "  target: dev\n"
        "  outputs:\n"
        "    dev:\n"
        "      type: duckdb\n"
        f"      path: {DUCKDB_PATH}\n"
    )

with open(f"{DBT_DIR}/models/sources.yml", "w") as f:
    f.write(
        "version: 2\n"
        "sources:\n"
        "  - name: raw_dane\n"
        "    schema: raw\n"
        "    tables:\n"
        "      - name: klienci\n"
        "      - name: produkty\n"
        "      - name: zamowienia\n"
        "      - name: zamowienia_streaming\n"
        "      - name: pozycje_zamowien\n"
        "      - name: pozycje_zamowien_streaming\n"
    )

with open(f"{DBT_DIR}/models/schema.yml", "w") as f:
    f.write(
        "version: 2\n"
        "models:\n"
        "  - name: dim_klienci\n"
        "    columns:\n"
        "      - name: klient_id\n"
        "        tests: [not_null]\n"
        "  - name: fact_sprzedaz\n"
        "    columns:\n"
        "      - name: pozycja_id\n"
        "        tests: [unique, not_null]\n"
        "      - name: klient_id\n"
        "        tests:\n"
        "          - relationships:\n"
        "              arguments:\n"
        "                to: ref('dim_klienci')\n"
        "                field: klient_id\n"
    )

print(f"Szkielet projektu dbt gotowy: {DBT_DIR}")

In [ ]:
# --- Modele staging: czyszczenie + laczenie zrodel wsadowych i streamingowych ---

with open(f"{DBT_DIR}/models/staging/stg_klienci.sql", "w") as f:
    f.write('''{{ config(tags=['staging']) }}
select
    klient_id,
    upper(left(miasto, 1)) || lower(substr(miasto, 2)) as miasto,
    segment,
    dni_od_rejestracji
from {{ source('raw_dane', 'klienci') }}
''')

with open(f"{DBT_DIR}/models/staging/stg_produkty.sql", "w") as f:
    f.write('''{{ config(tags=['staging']) }}
select produkt_id, kategoria, cena
from {{ source('raw_dane', 'produkty') }}
''')

with open(f"{DBT_DIR}/models/staging/stg_zamowienia.sql", "w") as f:
    f.write('''{{ config(tags=['staging']) }}
-- Laczymy zamowienia zaladowane wsadowo z tymi, ktore dotarly przez strumien (Modul 9) --
-- z punktu widzenia modelu gwiazdy to jedno, spojne zrodlo zamowien.
select zamowienie_id, klient_id, data_zamowienia, status
from {{ source('raw_dane', 'zamowienia') }}
union all
select zamowienie_id, klient_id, data_zamowienia, status
from {{ source('raw_dane', 'zamowienia_streaming') }}
''')

with open(f"{DBT_DIR}/models/staging/stg_pozycje_zamowien.sql", "w") as f:
    f.write('''{{ config(tags=['staging']) }}
select pozycja_id, zamowienie_id, produkt_id, ilosc, cena_jednostkowa, rabat
from {{ source('raw_dane', 'pozycje_zamowien') }}
union all
select pozycja_id, zamowienie_id, produkt_id, ilosc, cena_jednostkowa, rabat
from {{ source('raw_dane', 'pozycje_zamowien_streaming') }}
''')

print("Modele staging zapisane.")

In [ ]:
# --- Snapshot SCD2 wymiaru klientow (Modul 7) ---

with open(f"{DBT_DIR}/snapshots/snap_klienci.sql", "w") as f:
    f.write('''{% snapshot snap_klienci %}
{{
    config(
        target_schema='main',
        unique_key='klient_id',
        strategy='check',
        check_cols=['miasto', 'segment'],
    )
}}
select * from {{ ref('stg_klienci') }}
{% endsnapshot %}
''')

print("Snapshot SCD2 zapisany.")

In [ ]:
# --- Model gwiazdy: wymiary + tabela faktow (Modul 6) ---

with open(f"{DBT_DIR}/models/gwiazda/dim_klienci.sql", "w") as f:
    f.write('''{{ config(tags=['gwiazda']) }}
-- Tylko biezaco aktualne wiersze z historii SCD2
select klient_id, miasto, segment, dni_od_rejestracji, dbt_valid_from, dbt_valid_to
from {{ ref('snap_klienci') }}
where dbt_valid_to is null
''')

with open(f"{DBT_DIR}/models/gwiazda/dim_produkty.sql", "w") as f:
    f.write('''{{ config(tags=['gwiazda']) }}
select produkt_id, kategoria, cena
from {{ ref('stg_produkty') }}
''')

with open(f"{DBT_DIR}/models/gwiazda/dim_data.sql", "w") as f:
    f.write('''{{ config(tags=['gwiazda']) }}
with kalendarz as (
    select unnest(generate_series(DATE '2023-01-01', DATE '2026-12-31', INTERVAL 1 DAY)) as data
)
select
    data as data_id,
    extract(year from data) as rok,
    extract(quarter from data) as kwartal,
    extract(month from data) as miesiac,
    extract(day from data) as dzien,
    case when extract(dow from data) in (0, 6) then true else false end as czy_weekend
from kalendarz
''')

with open(f"{DBT_DIR}/models/gwiazda/fact_sprzedaz.sql", "w") as f:
    f.write('''{{ config(tags=['gwiazda']) }}
-- Ziarno: jedna pozycja zamowienia
select
    p.pozycja_id,
    z.zamowienie_id,
    z.klient_id,
    p.produkt_id,
    z.data_zamowienia as data_id,
    z.status,
    p.ilosc,
    p.cena_jednostkowa,
    p.rabat,
    p.ilosc * p.cena_jednostkowa * (1 - p.rabat) as wartosc_netto
from {{ ref('stg_pozycje_zamowien') }} p
join {{ ref('stg_zamowienia') }} z on p.zamowienie_id = z.zamowienie_id
''')

print("Modele gwiazdy zapisane.")

## 3. Skrypty potoku: generowanie danych, streaming, raport

Airflow będzie orkiestrować **zewnętrzne skrypty**, a nie kod uruchamiany bezpośrednio w
procesie schedulera (dokładnie jak w sekcji "wywołanie zewnętrzne" z Modułu 3). Zapisujemy
teraz cztery skrypty na dysk:

1. `generuj_dane_koncowe.py` -- tworzy nasz znany zestaw danych sklepu internetowego i
   ładuje go do schematu `raw` w DuckDB (uruchamiany głównym Pythonem -- ma `pandas`,
   `numpy`, `duckdb`).
2. `symuluj_nowe_zamowienia.py` -- symuluje nowe zamówienia "docierające z zewnątrz" do
   strefy lądowania (pliki JSON) -- w prawdziwym systemie zastąpiłby to temat Kafki
   (Moduł 8).
3. `przetworz_strumien_koncowy.py` -- zadanie Spark Structured Streaming (Moduł 9), które
   odczytuje strefę lądowania i dopisuje nowe zamówienia do `raw.zamowienia_streaming` /
   `raw.pozycje_zamowien_streaming` (uruchamiany Pythonem z `/tmp/spark_venv`).
4. `raport_koncowy.py` -- finalne zapytanie analityczne na gotowej hurtowni (główny Python).

> ⚠️ **BashOperator uruchamia się we własnym, tymczasowym katalogu**
>
> Domyślnie `BashOperator` wykonuje polecenie w **nowo utworzonym katalogu tymczasowym**, a nie w katalogu, z którego uruchomiono ten notebook! Jeśli skrypt polega na `os.path.abspath(".")`, żeby znaleźć swoje pliki, w zadaniu Airflow znajdzie zupełnie inny, pusty katalog. Rozwiązanie: przekazujemy jawnie `cwd=...` do każdego `BashOperator`-a **oraz** wszystkie ścieżki w naszych skryptach budujemy jako bezwzględne, znane już w momencie zapisu skryptu na dysk (nie polegając na katalogu roboczym w czasie wykonania).

In [ ]:
GLOWNY_PYTHON = os.environ.get("PYTHON_GLOWNY") or __import__("sys").executable
SPARK_PYTHON = "/tmp/spark_venv/bin/python3"

kod_generatora = f'''import os
import numpy as np
import pandas as pd
import duckdb

DUCKDB_PATH = {DUCKDB_PATH!r}

rng = np.random.default_rng(7)

n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
klienci_pd = pd.DataFrame({{
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=[0.35, 0.45, 0.20]),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
}})

n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {{"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}}
kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({{
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2) for k in kategorie_produktow],
}})

n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values
prawdop_anulowania = np.where(segment_zamowien == "Nowy", 0.22, np.where(segment_zamowien == "Standardowy", 0.10, 0.04))
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(anulowane == 1, "Anulowane", rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]))
data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)
zamowienia_pd = pd.DataFrame({{
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
}})

wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]
for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({{
            "pozycja_id": pozycja_id, "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id), "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]), "rabat": float(rabat),
        }})
        pozycja_id += 1
pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

con = duckdb.connect(DUCKDB_PATH)
con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE OR REPLACE TABLE raw.klienci AS SELECT * FROM klienci_pd")
con.execute("CREATE OR REPLACE TABLE raw.produkty AS SELECT * FROM produkty_pd")
con.execute("CREATE OR REPLACE TABLE raw.zamowienia AS SELECT * FROM zamowienia_pd")
con.execute("CREATE OR REPLACE TABLE raw.pozycje_zamowien AS SELECT * FROM pozycje_zamowien_pd")
con.execute(
    "CREATE OR REPLACE TABLE raw.zamowienia_streaming ("
    "zamowienie_id INTEGER, klient_id INTEGER, data_zamowienia TIMESTAMP, status VARCHAR)"
)
con.execute(
    "CREATE OR REPLACE TABLE raw.pozycje_zamowien_streaming ("
    "pozycja_id INTEGER, zamowienie_id INTEGER, produkt_id INTEGER, "
    "ilosc INTEGER, cena_jednostkowa DOUBLE, rabat DOUBLE)"
)
con.close()
print(f"Wygenerowano dane bazowe: {{len(klienci_pd)}} klientow, {{len(zamowienia_pd)}} zamowien, {{len(pozycje_zamowien_pd)}} pozycji.")
'''

with open(f"{PROJEKT_DIR}/generuj_dane_koncowe.py", "w") as f:
    f.write(kod_generatora)

print("Zapisano generuj_dane_koncowe.py")

In [ ]:
LANDING_ZAM = f"{PROJEKT_DIR}/landing_zamowien"
LANDING_POZ = f"{PROJEKT_DIR}/landing_pozycje"

kod_symulatora = f'''import os
import json
import numpy as np

LANDING_ZAM = {LANDING_ZAM!r}
LANDING_POZ = {LANDING_POZ!r}
os.makedirs(LANDING_ZAM, exist_ok=True)
os.makedirs(LANDING_POZ, exist_ok=True)

rng = np.random.default_rng(123)

nowe_zamowienia = []
nowe_pozycje = []
nastepny_pozycja_id = 900000
for i in range(5):
    zamowienie_id = 100000 + i
    nowe_zamowienia.append({{
        "zamowienie_id": zamowienie_id,
        "klient_id": int(rng.integers(1, 300)),
        "data_zamowienia": "2026-09-25T12:00:00",
        "status": "Zlozone",
    }})
    for _ in range(int(rng.integers(1, 3))):
        nowe_pozycje.append({{
            "pozycja_id": nastepny_pozycja_id,
            "zamowienie_id": zamowienie_id,
            "produkt_id": int(rng.integers(1, 40)),
            "ilosc": int(rng.integers(1, 4)),
            "cena_jednostkowa": round(float(rng.uniform(20, 500)), 2),
            "rabat": 0.0,
        }})
        nastepny_pozycja_id += 1

suffix = os.urandom(4).hex()
with open(f"{{LANDING_ZAM}}/paczka_{{suffix}}.json", "w") as f:
    for w in nowe_zamowienia:
        f.write(json.dumps(w) + "\\n")
with open(f"{{LANDING_POZ}}/paczka_{{suffix}}.json", "w") as f:
    for w in nowe_pozycje:
        f.write(json.dumps(w) + "\\n")

print(f"Zapisano {{len(nowe_zamowienia)}} nowych zamowien i {{len(nowe_pozycje)}} pozycji (paczka {{suffix}})")
'''

with open(f"{PROJEKT_DIR}/symuluj_nowe_zamowienia.py", "w") as f:
    f.write(kod_symulatora)

print("Zapisano symuluj_nowe_zamowienia.py")

In [ ]:
CHECKPOINT_ZAM = f"{PROJEKT_DIR}/checkpoint_streaming_zamowienia"
CHECKPOINT_POZ = f"{PROJEKT_DIR}/checkpoint_streaming_pozycje"

kod_streamingu = f'''import duckdb
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

LANDING_ZAM = {LANDING_ZAM!r}
LANDING_POZ = {LANDING_POZ!r}
CHECKPOINT_ZAM = {CHECKPOINT_ZAM!r}
CHECKPOINT_POZ = {CHECKPOINT_POZ!r}
DUCKDB_PATH = {DUCKDB_PATH!r}

_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)
spark = (
    SparkSession.builder.appName("potok_koncowy_streaming")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

schemat_zamowien = StructType([
    StructField("zamowienie_id", IntegerType()),
    StructField("klient_id", IntegerType()),
    StructField("data_zamowienia", TimestampType()),
    StructField("status", StringType()),
])
schemat_pozycji = StructType([
    StructField("pozycja_id", IntegerType()),
    StructField("zamowienie_id", IntegerType()),
    StructField("produkt_id", IntegerType()),
    StructField("ilosc", IntegerType()),
    StructField("cena_jednostkowa", DoubleType()),
    StructField("rabat", DoubleType()),
])


def zapisz_do_tabeli(nazwa_tabeli):
    def _zapisz(batch_df, batch_id):
        if batch_df.rdd.isEmpty():
            print(f"[{{nazwa_tabeli}}] mikro-wsad {{batch_id}}: brak nowych danych.")
            return
        pdf = batch_df.toPandas()
        con = duckdb.connect(DUCKDB_PATH)
        con.execute(f"INSERT INTO {{nazwa_tabeli}} SELECT * FROM pdf")
        con.close()
        print(f"[{{nazwa_tabeli}}] mikro-wsad {{batch_id}}: doladowano {{len(pdf)}} wierszy")
    return _zapisz


strumien_zam = spark.readStream.schema(schemat_zamowien).json(LANDING_ZAM)
q1 = (
    strumien_zam.writeStream
    .foreachBatch(zapisz_do_tabeli("raw.zamowienia_streaming"))
    .option("checkpointLocation", CHECKPOINT_ZAM)
    .trigger(availableNow=True)
    .start()
)
q1.awaitTermination()

strumien_poz = spark.readStream.schema(schemat_pozycji).json(LANDING_POZ)
q2 = (
    strumien_poz.writeStream
    .foreachBatch(zapisz_do_tabeli("raw.pozycje_zamowien_streaming"))
    .option("checkpointLocation", CHECKPOINT_POZ)
    .trigger(availableNow=True)
    .start()
)
q2.awaitTermination()

spark.stop()
print("Zadanie streamingowe zakonczone.")
'''

with open(f"{PROJEKT_DIR}/przetworz_strumien_koncowy.py", "w") as f:
    f.write(kod_streamingu)

print("Zapisano przetworz_strumien_koncowy.py")

> 🏭 **duckdb w środowisku Sparka**
>
> Zadanie streamingowe zapisuje bezpośrednio do pliku DuckDB używając połączenia `duckdb`, dlatego środowisko `/tmp/spark_venv` musi mieć zainstalowany pakiet `duckdb` (nie tylko `pyspark`). Instalujemy go raz poniżej -- na produkcji znalazłby się w obrazie kontenera/klastra, na którym uruchamiane są zadania Sparka.

In [ ]:
wynik_pip = subprocess.run(
    ["/tmp/spark_venv/bin/pip", "install", "duckdb", "--quiet"],
    capture_output=True, text=True,
)
print(wynik_pip.stdout[-500:] or "duckdb juz zainstalowany lub instalacja zakonczona bez wyjscia.")

In [ ]:
kod_raportu = f'''import duckdb

con = duckdb.connect({DUCKDB_PATH!r})

print("=== Raport koncowy potoku danych ===\\n")

zapytanie_miasta = (
    "select k.miasto, round(sum(f.wartosc_netto), 2) as suma_sprzedazy, count(*) as liczba_pozycji "
    "from fact_sprzedaz f join dim_klienci k on f.klient_id = k.klient_id "
    "group by k.miasto order by suma_sprzedazy desc"
)
print("-- Sprzedaz wg miasta --")
print(con.execute(zapytanie_miasta).df())

zapytanie_zrodlo = (
    "select case when zamowienie_id >= 100000 then 'streaming' else 'wsad (batch)' end as zrodlo, "
    "count(distinct zamowienie_id) as liczba_zamowien, count(*) as liczba_pozycji "
    "from fact_sprzedaz group by 1"
)
print("\\n-- Zamowienia wg zrodla (wsad vs strumien) --")
print(con.execute(zapytanie_zrodlo).df())

con.close()
print("\\nRaport zakonczony.")
'''

with open(f"{PROJEKT_DIR}/raport_koncowy.py", "w") as f:
    f.write(kod_raportu)

print("Zapisano raport_koncowy.py")

## 4. Orkiestracja: DAG spinający cały potok

Teraz łączymy wszystko w jeden DAG Airflow. Kolejność zadań odzwierciedla logiczną
kolejność potoku:

1. `wygeneruj_dane_bazowe` -- ładuje wsadowy zestaw danych.
2. `symuluj_nowe_zamowienia` -- symuluje nowe zdarzenia docierające "z zewnątrz".
3. `czekaj_na_nowe_zamowienia` -- `PythonSensor` (Moduł 3), który upewnia się, że dane
   rzeczywiście czekają w strefie lądowania, zanim uruchomimy kosztowne zadanie Sparka.
4. `przetworz_strumien` -- zadanie Spark Structured Streaming ładujące nowe dane do `raw`.
5. `dbt_uruchom_staging` → `dbt_snapshot_klienci` → `dbt_uruchom_gwiazde` → `dbt_testy` --
   pełny potok transformacji dbt (Moduł 4-7), w poprawnej kolejności zależności.
6. `raport_koncowy` -- finalny raport analityczny.

In [ ]:
cmd_generuj = f"{GLOWNY_PYTHON} {PROJEKT_DIR}/generuj_dane_koncowe.py"
cmd_symuluj = f"{GLOWNY_PYTHON} {PROJEKT_DIR}/symuluj_nowe_zamowienia.py"
cmd_streaming = f"{SPARK_PYTHON} {PROJEKT_DIR}/przetworz_strumien_koncowy.py"
cmd_dbt_staging = f"dbt run --select tag:staging --project-dir {DBT_DIR} --profiles-dir {DBT_DIR}"
cmd_dbt_snapshot = f"dbt snapshot --project-dir {DBT_DIR} --profiles-dir {DBT_DIR}"
cmd_dbt_gwiazda = f"dbt run --select tag:gwiazda --project-dir {DBT_DIR} --profiles-dir {DBT_DIR}"
cmd_dbt_test = f"dbt test --project-dir {DBT_DIR} --profiles-dir {DBT_DIR}"
cmd_raport = f"{GLOWNY_PYTHON} {PROJEKT_DIR}/raport_koncowy.py"

dag_kod = f'''from datetime import datetime
import os

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.sensors.python import PythonSensor


def sprawdz_nowe_zamowienia():
    return len(os.listdir({LANDING_ZAM!r})) > 0


with DAG(
    dag_id="potok_koncowy_pl",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    wygeneruj_dane = BashOperator(
        task_id="wygeneruj_dane_bazowe", bash_command={cmd_generuj!r}, cwd={PROJEKT_DIR!r},
    )
    symuluj_zdarzenia = BashOperator(
        task_id="symuluj_nowe_zamowienia", bash_command={cmd_symuluj!r}, cwd={PROJEKT_DIR!r},
    )
    czekaj_na_dane = PythonSensor(
        task_id="czekaj_na_nowe_zamowienia",
        python_callable=sprawdz_nowe_zamowienia,
        poke_interval=2,
        timeout=30,
    )
    przetworz_strumien = BashOperator(
        task_id="przetworz_strumien", bash_command={cmd_streaming!r}, cwd={PROJEKT_DIR!r},
    )
    dbt_staging = BashOperator(
        task_id="dbt_uruchom_staging", bash_command={cmd_dbt_staging!r}, cwd={PROJEKT_DIR!r},
    )
    dbt_snap = BashOperator(
        task_id="dbt_snapshot_klienci", bash_command={cmd_dbt_snapshot!r}, cwd={PROJEKT_DIR!r},
    )
    dbt_gwiazda = BashOperator(
        task_id="dbt_uruchom_gwiazde", bash_command={cmd_dbt_gwiazda!r}, cwd={PROJEKT_DIR!r},
    )
    dbt_testy = BashOperator(
        task_id="dbt_testy", bash_command={cmd_dbt_test!r}, cwd={PROJEKT_DIR!r},
    )
    raport = BashOperator(
        task_id="raport_koncowy", bash_command={cmd_raport!r}, cwd={PROJEKT_DIR!r},
    )

    (
        wygeneruj_dane
        >> symuluj_zdarzenia
        >> czekaj_na_dane
        >> przetworz_strumien
        >> dbt_staging
        >> dbt_snap
        >> dbt_gwiazda
        >> dbt_testy
        >> raport
    )
'''

with open(f"{AIRFLOW_HOME}/dags/potok_koncowy_dag.py", "w") as f:
    f.write(dag_kod)

print("DAG potok_koncowy_pl zapisany.")

> ⚠️ **Sprzątanie przed uruchomieniem**
>
> Jeśli uruchamiasz ten notebook wielokrotnie, warto usunąć stan z poprzedniego przebiegu (bazę DuckDB, checkpointy Sparka, strefę lądowania) -- inaczej `dbt` zobaczy dane z poprzedniego uruchomienia zmieszane z nowymi. Poniższa komórka robi to za Ciebie.

In [ ]:
import shutil

for sciezka in [
    DUCKDB_PATH, f"{DUCKDB_PATH}.wal",
    LANDING_ZAM, LANDING_POZ,
    CHECKPOINT_ZAM, CHECKPOINT_POZ,
    f"{DBT_DIR}/target", f"{DBT_DIR}/logs",
]:
    if os.path.isdir(sciezka):
        shutil.rmtree(sciezka, ignore_errors=True)
    elif os.path.exists(sciezka):
        os.remove(sciezka)

print("Wyczyszczono stan z poprzednich przebiegow.")

In [ ]:
wynik = airflow("dags", "test", "potok_koncowy_pl", "2024-01-01")
print("\nDAG zakonczony sukcesem." if wynik.returncode == 0 else "DAG zakonczony bledem.")

## 5. Weryfikacja wyniku

DAG zakończył się bez błędów -- ale sprawdźmy naprawdę, co wylądowało w naszej hurtowni.
Odpytujemy tę samą bazę DuckDB bezpośrednio z notebooka.

In [ ]:
import duckdb

con = duckdb.connect(DUCKDB_PATH)

print("-- dim_klienci: liczba aktualnych wierszy --")
print(con.execute("select count(*) as liczba_klientow from dim_klienci").df())

print("\n-- fact_sprzedaz: sprzedaz wg miasta --")
print(con.execute('''
    select k.miasto, round(sum(f.wartosc_netto), 2) as suma_sprzedazy, count(*) as liczba_pozycji
    from fact_sprzedaz f
    join dim_klienci k on f.klient_id = k.klient_id
    group by k.miasto
    order by suma_sprzedazy desc
''').df())

print("\n-- Zamowienia wg zrodla (dowod, ze streaming naprawde zadzialal) --")
print(con.execute('''
    select case when zamowienie_id >= 100000 then 'streaming' else 'wsad (batch)' end as zrodlo,
           count(distinct zamowienie_id) as liczba_zamowien,
           count(*) as liczba_pozycji
    from fact_sprzedaz
    group by 1
''').df())

con.close()

> 💡 **Co właśnie zweryfikowaliśmy**
>
> Wiersze ze źródła `streaming` w tabeli faktów dowodzą, że dane, które `symuluj_nowe_zamowienia.py` zapisał jako pliki JSON, zostały odczytane przez Spark Structured Streaming, doładowane do `raw.zamowienia_streaming` / `raw.pozycje_zamowien_streaming`, a następnie przekształcone przez dbt (`UNION ALL` w modelach staging) w normalne wiersze tabeli faktów -- **cały potok z sekcji 0 zadziałał od początku do końca, automatycznie, jednym poleceniem `airflow dags test`.**

## 6. Podsumowanie całego kursu

Przebyliśmy długą drogę od definicji inżynierii danych po działający potok end-to-end:

- **Moduł 1** -- czym jest inżynieria danych, ETL vs ELT, anatomia nowoczesnego stosu danych.
- **Moduł 2-3** -- Apache Airflow: DAG-i, operatory, zależności, XCom, sensory, branching,
  dynamiczne mapowanie zadań, retry i trigger rules.
- **Moduł 4-5** -- dbt: `source()`/`ref()`, materializacje, testy, dokumentacja, makra,
  seedy, modele przyrostowe.
- **Moduł 6** -- modelowanie danych: Kimball vs Inmon, fakty i wymiary, schemat gwiazdy,
  ziarno tabeli faktów.
- **Moduł 7** -- Slowly Changing Dimensions: SCD Type 1 vs Type 2, implementacja przez
  `dbt snapshot`.
- **Moduł 8** -- podstawy Apache Kafka: tematy, partycje, grupy konsumenckie, gwarancje
  dostarczenia.
- **Moduł 9** -- Spark Structured Streaming: model mikro-wsadowy, okna czasowe, watermark,
  `foreachBatch`.
- **Moduł 10** -- **ten moduł**: wszystko powyżej połączone w jeden, samodzielnie
  uruchamiany potok danych.

Gratulacje -- to solidny fundament pod pracę jako inżynier danych. Świat narzędzi wokół
tego stosu wciąż się zmienia (Iceberg/Delta Lake jako otwarte formaty tabel, silniki takie
jak Flink dla prawdziwego przetwarzania rekord-po-rekordzie, platformy typu Databricks czy
Snowflake jako zarządzane środowiska uruchomieniowe), ale **fundamentalne pojęcia, których
się nauczyłeś -- orkiestracja, transformacje jako kod, modelowanie wymiarowe, streaming --
przenoszą się bezpośrednio na każde z tych narzędzi.**

### 📝 Ćwiczenie 1: Nowy wymiar produktów

Wzoruj się na `dim_klienci` (SCD2) i zaprojektuj (opisz kroki, nie musisz pisać pełnego kodu) jak zamienić `dim_produkty` na wymiar SCD2 -- tak, aby zmiana `cena` produktu była śledzona historycznie zamiast nadpisywana. Jakie `check_cols` ustawiłbyś w snapshotcie?

<details>
<summary>Pokaż rozwiązanie</summary>

```text
1. Utworzyc snapshot `snap_produkty` analogiczny do `snap_klienci`, z
   check_cols=['cena', 'kategoria'] (dowolna zmiana ktorejs z tych kolumn tworzy nowa
   wersje historyczna).
2. Zmienic `dim_produkty.sql` tak, zeby czytal z `{{ ref('snap_produkty') }}` zamiast
   bezposrednio ze stagingu, filtrujac `dbt_valid_to is null` (tylko aktualne wiersze).
3. W fact_sprzedaz.sql doloczyc join po produkt_id + zakres dat wazny w momencie
   zamowienia (jak w Module 7: data_zamowienia >= dbt_valid_from AND (dbt_valid_to IS
   NULL OR data_zamowienia < dbt_valid_to)), jesli chcemy analizowac sprzedaz z cena
   obowiazujaca w chwili zamowienia, a nie cena aktualna.
```

</details>

### 📝 Ćwiczenie 2: Harmonogram produkcyjny

DAG w tym module ma `schedule=None` (uruchamiamy go ręcznie). Zaproponuj sensowny harmonogram cron dla tego potoku na produkcji, zakładając, że: nowe zamówienia streamingowe powinny być przetwarzane co 15 minut, ale pełne przebudowanie modelu gwiazdy (łącznie z testami) wystarczy raz dziennie w nocy. Jak rozbiłbyś to na dwa osobne DAG-i?

<details>
<summary>Pokaż rozwiązanie</summary>

```text
DAG 1 -- "potok_streaming" (co 15 min, schedule='*/15 * * * *'):
    symuluj/odbierz nowe zamowienia -> przetworz_strumien -> dbt run --select tag:staging
    (tylko odswiezenie widokow stagingowych, tanie i szybkie)

DAG 2 -- "potok_dzienny" (raz w nocy, schedule='0 2 * * *'):
    dbt snapshot -> dbt run --select tag:gwiazda -> dbt test -> raport_koncowy

Rozdzielenie na dwa DAG-i odzwierciedla rozna czestotliwosc biznesowa: swiezosc danych
"live" (co 15 min) jest tania, bo modele staging to tylko widoki (view) bez przeliczania
calej hurtowni; kosztowniejsze przebudowanie modelu gwiazdy + testy odbywa sie rzadziej.
```

</details>

> 🔥 **Wyzwanie końcowe**
>
> Ten potok zakłada, że dane trafiające przez streaming zawsze mają odpowiadające im pozycje zamówienia w tej samej (albo szybciej przetworzonej) partii. W praktyce zdarzenia mogą docierać w różnej kolejności -- zamówienie może przyjść zanim jego pozycje zdążą dotrzeć (albo odwrotnie). Zaprojektuj (opisowo) rozwiązanie tego problemu w architekturze z tego modułu: jak upewnić się, że `fact_sprzedaz` nie zgubi *trwale* pozycji zamówienia, którego jeszcze nie ma w `stg_zamowienia` w momencie danego przebiegu dbt?

<details>
<summary>Pokaż rozwiązanie</summary>

```text
Kluczowa obserwacja: fact_sprzedaz uzywa INNER JOIN miedzy pozycjami a zamowieniami --
pozycja bez pasujacego zamowienia po prostu zniknie z wyniku PRZY TYM przebiegu, ale
NIE jest trwale utracona (dane nadal siedza w raw/staging). Kolejny przebieg dbt (ten sam
model, run po run) zlapie ja, gdy tylko zamowienie tez dotrze -- poniewaz kazdy `dbt run`
przelicza widoki od nowa z aktualnego stanu zrodel.

Ryzyko pojawia sie tylko, gdyby fact_sprzedaz byla materializowana jako INCREMENTAL model
z logika "wstaw i nigdy nie wracaj" (Modul 5) -- wtedy spozniona pozycja NIGDY by nie
zostala domodelowana, bo incremental model nie przegląda ponownie starych partycji.
Rozwiazanie: albo (a) zostawic fact_sprzedaz jako zwykly `view`/`table` (pelne przeliczenie
za kazdym razem -- co robimy w tym module), co jest bezpieczne kosztem wydajnosci przy
duzej skali, albo (b) jesli potrzebny jest model incremental, dodac logike "late-arriving
facts": trzymac osobno tabele "sierot" (pozycje bez zamowienia) i doliczac je w kolejnym
przebiegu, gdy zamowienie sie pojawi (typowy wzorzec w hurtowniach czasu rzeczywistego).
```

</details>

## Gratulacje!

Ukończyłeś kurs Inżynierii Danych -- od pierwszych zasad, przez Airflow i dbt, po
modelowanie wymiarowe i streaming, aż po samodzielnie działający potok end-to-end. To,
czego się nauczyłeś, to nie tylko konkretne narzędzia, ale **sposób myślenia o danych jako
o produkcie**: niezawodnym, testowanym, zorkiestrowanym i gotowym do zmian w czasie.

Powodzenia w dalszej pracy z danymi!